# Data analysis for ingest pipeline

Validate that TasteMender's imported audio-feature data is complete, internally consistent, and faithful to the original AcousticBrainz source data.

This work should identify any issues introduced during ingestion, transformation, normalization, or storage, and establish whether each feature provides enough useful variation to support recommendation ranking.

In [ ]:
import os, uuid, sys, random
import django
import numpy as np
import numpy.typing as npt
import pandas as pd
import matplotlib.pyplot as plt

# Enable access to Django project inside current notebook
PROJECT_PATH = os.path.dirname(os.path.abspath(""))
sys.path.append(PROJECT_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "music_recommendation.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from recommend_api.models import GenreDortmund, GenreRosamerica, Track, Artist, Album

### Display genre distribution charts for 2 classifiers: Dortmund and Rosamerica

In [ ]:
feature_store = np.load("../features_and_index.npz", allow_pickle=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Pull genre codes for all tracks into a pandas Series
dort = pd.Series(feature_store["genre_dortmund"], name="count")
rosa = pd.Series(feature_store["genre_rosamerica"])

# Count and plot Dortmund, genre labels are stored in Django models
counts_dort = dort.value_counts()
dort_labels_by_code = dict(GenreDortmund.objects.values_list("code", "label"))
labels_dort = [dort_labels_by_code[code] for code in counts_dort.index]

axes[0].pie(counts_dort, labels=labels_dort, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Genre distribution (Dortmund)")
axes[0].legend(labels_dort, loc="best", fontsize="small")

# Count and plot Rosamerica, genre labels are stored in Django models
counts_rosa = rosa.value_counts()
rosa_labels_by_code = dict(GenreRosamerica.objects.values_list("code", "label"))
labels_rosa = [
    rosa_labels_by_code[code] for code in counts_rosa.index
]
axes[1].pie(counts_rosa, labels=labels_rosa, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Genre distribution (Rosamerica)")
axes[1].legend(labels_rosa, loc="best", fontsize="small")

plt.tight_layout()
plt.show()

The Dortmund classification designates a whopping 77.8% as being electronic music while Rosamerica splits up genres more evenly. The official AcousticBrainz [accuracy metrics](https://acousticbrainz.org/datasets/accuracy) put Dortmund at 60.25% and Rosamerica at 87.56%.

We ca safely conclude that the Rosamerica classification is the best choice for determining musical genre in TasteMender due to its higher accuracy.

### Feature Distribution - Normalized vs Raw
We'll compare data from the two feature matrices to see how each audio feature maps to the standard distribution. 

- Does normalization change the shape of the distribution?
- How are features distributed in general?

### Missing Data

#### Missing Release Year

In [ ]:
years: npt.NDArray[np.int16] = np.array(feature_store["years"])
num_tracks = len(years)
zero_years = np.sum(years == 0)

print(f"Tracks missing a release year: {zero_years:,} / {num_tracks:,}")

mbids_raw: npt.NDArray[np.void] = feature_store["mbids"]
zero_year_mbids_raw = random.sample(mbids_raw[years == 0].tolist(), 10)
zero_year_mbids = [str(uuid.UUID(bytes=mbid)) for mbid in zero_year_mbids_raw]

zero_year_tracks_qs = (
    Track.objects.filter(musicbrainz_recordingid__in=zero_year_mbids)
    .select_related("album")
    .prefetch_related("artists")
)

zero_year_tracks_df = pd.DataFrame(
    [
        (
            track.musicbrainz_recordingid,
            track.title,
            ", ".join(artist.name for artist in track.artists.all()),
            track.album.name if track.album else "<no album>",
            track.album.date if track.album else None,
        )
        for track in zero_year_tracks_qs
    ],
    columns=["mbid", "title", "artists", "album", "album_date"],
)

print("Sample of tracks missing a release year:")
display(zero_year_tracks_df)